# MaxSim convergence + sensitivity diagnostic (D-031 follow-up)

The Gate-3 primary result had MaxSim land ~0.075 macro-MRR **below** pooled (not at
parity) and fail the mean-collapse control. That *could* be a real null, or MaxSim
may be **undertrained under the locked shared hyperparameters** (temp 0.05 / lr 1e-3
were natural for pooled cosine; MaxSim's mean-of-max-cosine scores live on a
different scale). This is a **diagnostic, not a protocol change** -- it does not
revise the locked primary number (lock 13). It trains MaxSim-only on **one fold +
one seed** across a temperature x lr grid and reports, per cell: confirmation
macro-MRR, the mean-collapse gap, delta vs a same-fold pooled reference, and the
loss + dev-MRR traces (so we can see whether it plateaued).

**Read-out:** if MaxSim stays <= pooled at its own best temp/lr -> the null is
robust (report as-is). If a MaxSim-specific temp/lr reaches parity -> the primary is
reported as an equivalence null plus a hyperparameter-sensitivity note. Attach the
same three datasets as the primary run. Held-out test is never touched.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '9e47f2a04cbf5dfc9d652e3c36651a8d00493852'
WORKTREE = '/kaggle/working/SemKey'
FOLD = '0'                                  # one fold -- diagnostic, not the campaign
SEED = 20260722                             # one seed (lock 4/17)
EPOCHS, BATCH_SIZE, GRAD_CLIP = 40, 64, 1.0             # locked schedule (lock 17)
BASE_LR, BASE_TEMPERATURE = 1e-3, 0.05                  # the locked shared hyperparameters
SELECT_EVERY = 80
POOL_SIZE = 24
TEMPERATURES = (0.02, 0.05, 0.1, 0.2, 0.5)             # 0.05 is the locked value
LRS = (None, 3e-3)                          # None => BASE_LR
assert len(COMMIT) == 40

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as h:
    h.write("#!/usr/bin/env python3\nimport os, sys\np = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in p else 'x-access-token')\n")
os.chmod(askpass, 0o700)
cenv = os.environ.copy(); cenv.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE): shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=cenv)
finally:
    os.remove(askpass); del github_token, cenv
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
env = os.environ.copy(); env['PYTHONPATH'] = WORKTREE
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'evaluation.test_maxsim_convergence_diag'], check=True, cwd=WORKTREE, env=env)
sys.path.insert(0, WORKTREE)
print({'clone': 'PASS', 'self_tests': 'PASS'})

In [ ]:
def find_one(pattern):
    hits = glob.glob('/kaggle/input/**/' + pattern, recursive=True)
    assert len(hits) == 1, ('need exactly one ' + pattern, hits)
    return hits[0]

def find_text_dir():
    dirs = [Path(os.path.dirname(h)) for h in glob.glob('/kaggle/input/**/text_token_index.csv', recursive=True)]
    if len(dirs) == 1:
        return dirs[0]
    standalone = [d for d in dirs if not glob.glob(str(d.parent) + '/**/token_index.json', recursive=True)]
    assert len(standalone) == 1, ('cannot disambiguate text_token_index.csv', [str(d) for d in dirs])
    return standalone[0]

eeg_root = Path(os.path.dirname(find_one('token_index.json')))
text_root = find_text_dir()
protocol_root = Path(os.path.dirname(find_one('candidate_pools.csv')))
_tman = json.load(open(text_root / 'text_token_manifest.json', encoding='utf-8'))
print({'eeg_root': str(eeg_root), 'text_root': str(text_root), 'protocol_root': str(protocol_root),
       'text_combined_chunk_sha256': _tman.get('combined_chunk_sha256')})

In [ ]:
from evaluation.token_campaign_io import (
    load_eeg_lookup, load_text_lookup, load_assignments, load_candidate_pools, load_donors)
eeg = load_eeg_lookup(eeg_root)
text = load_text_lookup(text_root)
assignments = load_assignments(protocol_root / 'outer_split_assignments.csv')
pools = load_candidate_pools(protocol_root / 'candidate_pools.csv')
donors = load_donors(protocol_root / 'confirmation_donors.csv')
print({'eeg_trials': len(eeg), 'texts': len(text), 'assignments': len(assignments)})
dev = torch.device('cuda')
text = {k: {kk: vv.to(dev) for kk, vv in v.items()} for k, v in text.items()}
print({'text_cache_on': str(dev), 'gpu_mem_gb': round(torch.cuda.memory_allocated() / 1e9, 2)})

In [ ]:
from evaluation.maxsim_convergence_diag import run_convergence_diagnostic
from evaluation.token_training import TrainConfig
base = TrainConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=BASE_LR,
                   temperature=BASE_TEMPERATURE, grad_clip=GRAD_CLIP)
diag = run_convergence_diagnostic(
    FOLD, SEED, assignments, pools, donors, eeg, text,
    base_config=base, temperatures=TEMPERATURES, lrs=LRS,
    select_every=SELECT_EVERY, device='cuda', pool_size=POOL_SIZE)
print('pooled reference (same fold):', diag['pooled_reference_mrr'])

In [ ]:
diag['project_commit'] = COMMIT
out = '/kaggle/working/maxsim_convergence_diag.json'
with open(out, 'w', encoding='utf-8') as h:
    json.dump(diag, h, indent=2, sort_keys=True); h.write('\n')
diag_sha = hashlib.sha256(open(out, 'rb').read()).hexdigest()
ref = diag['pooled_reference_mrr']
print('=== MaxSim CONVERGENCE / SENSITIVITY (fold', diag['fold'], 'seed', diag['seed'], ') ===')
print('pooled reference macro-MRR :', ref, '| locked temp', diag['locked_temperature'], 'lr', diag['locked_lr'])
print('%-6s %-7s %-8s %-9s %-8s %-7s  %s' % ('temp', 'lr', 'maxMRR', 'dVsPool', 'mcGap', 'parity', 'loss(first->last, tail)'))
for c in diag['grid']:
    L = c['loss']
    print('%-6s %-7s %-8s %-9s %-8s %-7s  %s->%s (tail %s)' % (
        c['temperature'], c['lr'], c['maxsim_confirmation_mrr'], c['delta_vs_pooled'],
        c['mean_collapse_gap'], c['reaches_parity'], L['first'], L['last'], L['tail_mean']))
print('best cell :', diag['best_cell'])
print('VERDICT   :', diag['verdict'])
print('diag sha256:', diag_sha)
print('saved', out)

Save `maxsim_convergence_diag.json` for provenance. Interpretation:

- **VERDICT = NULL_ROBUST** -> MaxSim stays below pooled even at its own best
  temp/lr: the primary negative is real; report the pooled-vs-token diagnostic as a
  clean null (consistent with ABPR) and move on -- no protocol change.
- **VERDICT = PARITY_REACHABLE** -> a MaxSim-specific temp/lr reaches pooled: the
  locked primary (shared hyperparameters) still stands as the headline, but report
  it as an *equivalence* null with this sensitivity analysis, not "late interaction
  is harmful". Also sanity-check that the loss/dev traces plateaued (not still
  descending at the last step) before trusting either verdict.

This is a diagnostic; the locked primary number is not revised (lock 13). Held-out
test stays sealed.